In [20]:
from imutils import paths
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from random import shuffle
import random
import timeit

from sklearn.preprocessing import LabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
from sklearn.utils import class_weight

from tensorflow.keras.utils import to_categorical
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import Xception
from tensorflow.keras.layers import AveragePooling2D, Dropout, Flatten, Dense, Input, BatchNormalization
from tensorflow.keras.models import Model, model_from_json
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import losses
from tensorflow.keras import datasets, layers, models
import tensorflow as tf

import tensorflow_addons as tfa

from tensorflow.keras.applications import imagenet_utils
from tensorflow.keras.applications.xception import decode_predictions

In [2]:
from pyFile import ARModel
from tensorflow.compat.v1 import ConfigProto
from tensorflow.compat.v1 import InteractiveSession
import numpy as np

import matplotlib.pyplot as plt

In [3]:
modelAvl = ["vgg16","vgg19","inception","xception","resnet50","resnet101","densenet","inceptionResnet"]
modelSel = ["inception", "xception" ]
modelOnly = ["xception"]
losses = ["bce", "cce", "focal", "kld"]
lossSel = ["bce", "focal"]
lossOnly = ["focal"]

In [4]:
def fix_gpu():
    config = ConfigProto()
    config.gpu_options.allow_growth = True
    session = InteractiveSession(config=config)

fix_gpu()

2022-10-28 11:29:02.558047: I tensorflow/core/platform/cpu_feature_guard.cc:142] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  SSE4.1 SSE4.2 AVX AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.
2022-10-28 11:29:02.560995: I tensorflow/compiler/jit/xla_gpu_device.cc:99] Not creating XLA devices, tf_xla_enable_xla_devices not set
2022-10-28 11:29:02.568414: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcuda.so.1
2022-10-28 11:29:02.613819: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-10-28 11:29:02.613975: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:01:00.0 na

In [5]:
model = ARModel()

In [6]:
(data, labels) = model.loadImages(r'/home/bishal/Research/Allergic-Rhinitis/Dataset/all/rotate', 
        plotType="R", classification="multiclass", colorMode="RGB", cleanImageF=True, resize=True,
        correctColor=False, contours=False, crop=True ,printImgDemo=False)

[INFO]: Trying to Read the images from  /home/bishal/Research/Allergic-Rhinitis/Dataset/all/rotate
Images found : 90


In [7]:
(data, labels) = model.prepareData(data, labels, weightedLossCalc=True)

[INFO]: Preparing Data
{'imgNumber': 111, 'dataInfo': 'R', 'classification': 'multiclass', 'imageCount': 90, 'imageType': 'R', 'classType': 'multiclass', 'colorMode': 'RGB', 'clean': True, 'crop': True, 'resize': True, 'colorCorrect': False, 'imgDim': (224, 224, 3)}


In [8]:
trainAug = model.setDataAugmentation(normalizeData=False, rotate=2, zoom=0.15, wShift=0.2, hShift=0.2, 
                                shear=0.15, hFlip=True, vFlip=False, generateImages=False)


[INFO]: Augmenting Data with - 
{'rotate': 2, 'zoom': 0.15, 'wShift': 0.2, 'hShift': 0.2, 'shear': 0.15, 'hFlip': True, 'vFlip': False}


In [9]:
#plt.imshow(trainX[54])

In [10]:
#(trainX, trainY, testX, testY) = model.setPartition(data, labels, testSize=0.20)

In [11]:
#testY = np.expand_dims(testY, 0)
#testX = np.expand_dims(testX, 0)

In [13]:
#testY

In [15]:
#testX.shape

In [16]:
currBModel = model.setBaseModel("xception")
                
currHModel = model.setHeadModel(currBModel, dropoutRate=0.5, activation="siren")
finalModel = model.initModel(currBModel, currHModel, baseTrainable=False)
model.setHyperParameters(learningRate = 1e-3, epochs = 200, batchSize = 8)
finalModel = model.compileModel(finalModel, loss="focal")

2022-10-28 11:29:37.435097: I tensorflow/compiler/jit/xla_cpu_device.cc:41] Not creating XLA devices, tf_xla_enable_xla_devices not set
2022-10-28 11:29:37.435321: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:941] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2022-10-28 11:29:37.435528: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1720] Found device 0 with properties: 
pciBusID: 0000:01:00.0 name: NVIDIA TITAN RTX computeCapability: 7.5
coreClock: 1.77GHz coreCount: 72 deviceMemorySize: 23.65GiB deviceMemoryBandwidth: 625.94GiB/s
2022-10-28 11:29:37.435585: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcudart.so.10.1
2022-10-28 11:29:37.435610: I tensorflow/stream_executor/platform/default/dso_loader.cc:49] Successfully opened dynamic library libcublas.so.10
2022-10-28 11:29:37.435628: I tensorflow/stream_executor/platform/defaul

[INFO]: Model Selected -  xception
[INFO]: Initializing Model
[INFO]: Hyperparameters Set
[INFO]: Compiling Model


In [40]:
y_hat = []
y = []
start = timeit.default_timer()
for i in range(10):
    print("#### Iter - %s ####"%str(i+1))
    trainX = np.delete(data, [i], axis=0)
    trainY = np.delete(labels, [i], axis=0)
    testX = np.expand_dims(data[i], 0)
    testY = np.expand_dims(labels[i], 0)
    y.append(labels[i])
    print("Sizes : ", trainX.shape, "--", trainY.shape)
    print("Sizes : ", testX.shape, "--", testY.shape)
    
    (H, finalModel) = model.startTraining(finalModel, trainAug, trainX, trainY, 
                                            testX, testY, weightedLoss=True, learningDecay=False, earlyStop=True)
    
    predIdxs = model.startTesting(testX, testY, finalModel, voting=30)
    y_hat.append(predIdxs[0])
stop = timeit.default_timer()

#### Iter - 1 ####
Sizes :  (89, 224, 224, 3) -- (89, 3)
Sizes :  (1, 224, 224, 3) -- (1, 3)
[INFO] Model Training
Epoch 1/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0762 - accuracy: 0.7407

Epoch 00001: loss improved from inf to 0.07623, saving model to AR_MODEL.hdf5
Epoch 2/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0598 - accuracy: 0.8025

Epoch 00002: loss improved from 0.07623 to 0.05979, saving model to AR_MODEL.hdf5
Epoch 3/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0500 - accuracy: 0.8765

Epoch 00003: loss improved from 0.05979 to 0.05002, saving model to AR_MODEL.hdf5
Epoch 4/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0991 - accuracy: 0.7531

Epoch 00004: loss did not improve from 0.05002
Epoch 5/200
11/11 [==============================] - 0s 39ms/step - loss: 0.0700 - accuracy: 0.8148

Epoch 00005: loss did not improve from 0.05002
Epoch 6/200
11/11 [=========================

Epoch 30/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0626 - accuracy: 0.7654

Epoch 00030: loss did not improve from 0.05002
Epoch 31/200
11/11 [==============================] - 1s 44ms/step - loss: 0.0638 - accuracy: 0.8295

Epoch 00031: loss did not improve from 0.05002
Epoch 32/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0803 - accuracy: 0.7531

Epoch 00032: loss did not improve from 0.05002
Epoch 33/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0769 - accuracy: 0.8148

Epoch 00033: loss did not improve from 0.05002
Epoch 34/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0748 - accuracy: 0.8025

Epoch 00034: loss did not improve from 0.05002
Epoch 35/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0842 - accuracy: 0.7407

Epoch 00035: loss did not improve from 0.05002
Epoch 36/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0655 - accuracy: 0.7778

Epoc

11/11 [==============================] - 0s 45ms/step - loss: 0.0677 - accuracy: 0.8025

Epoch 00059: loss did not improve from 0.04680
Epoch 60/200
11/11 [==============================] - 1s 43ms/step - loss: 0.0731 - accuracy: 0.8148

Epoch 00060: loss did not improve from 0.04680
Epoch 61/200
11/11 [==============================] - 0s 43ms/step - loss: 0.0662 - accuracy: 0.8025

Epoch 00061: loss did not improve from 0.04680
Epoch 62/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0715 - accuracy: 0.8025

Epoch 00062: loss did not improve from 0.04680
Epoch 63/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0737 - accuracy: 0.8272

Epoch 00063: loss did not improve from 0.04680
Epoch 64/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0968 - accuracy: 0.7531

Epoch 00064: loss did not improve from 0.04680
Epoch 65/200
11/11 [==============================] - 0s 43ms/step - loss: 0.0771 - accuracy: 0.7654

Epoch 00065: loss

11/11 [==============================] - 0s 43ms/step - loss: 0.0722 - accuracy: 0.8025

Epoch 00088: loss did not improve from 0.04680
Epoch 89/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0679 - accuracy: 0.7901

Epoch 00089: loss did not improve from 0.04680
Epoch 90/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0662 - accuracy: 0.8025

Epoch 00090: loss did not improve from 0.04680
Epoch 91/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0712 - accuracy: 0.8148

Epoch 00091: loss did not improve from 0.04680
Epoch 92/200
11/11 [==============================] - 0s 45ms/step - loss: 0.0672 - accuracy: 0.8148

Epoch 00092: loss did not improve from 0.04680
Epoch 93/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0626 - accuracy: 0.8025

Epoch 00093: loss did not improve from 0.04680
Epoch 94/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0772 - accuracy: 0.7901

Epoch 00094: loss

11/11 [==============================] - 0s 42ms/step - loss: 0.0718 - accuracy: 0.7778

Epoch 00117: loss did not improve from 0.04484
Epoch 118/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0741 - accuracy: 0.8025

Epoch 00118: loss did not improve from 0.04484
Epoch 119/200
11/11 [==============================] - 0s 38ms/step - loss: 0.0843 - accuracy: 0.7037

Epoch 00119: loss did not improve from 0.04484
Epoch 120/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0639 - accuracy: 0.8025

Epoch 00120: loss did not improve from 0.04484
Epoch 121/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0757 - accuracy: 0.7654

Epoch 00121: loss did not improve from 0.04484
Epoch 122/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0579 - accuracy: 0.7901

Epoch 00122: loss did not improve from 0.04484
Epoch 123/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0547 - accuracy: 0.8395

Epoch 00123

...

11/11 [==============================] - 0s 40ms/step - loss: 0.0579 - accuracy: 0.8395

Epoch 00077: loss did not improve from 0.04527
Epoch 78/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0642 - accuracy: 0.8642

Epoch 00078: loss did not improve from 0.04527
Epoch 79/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0646 - accuracy: 0.8148

Epoch 00079: loss did not improve from 0.04527
Epoch 80/200
11/11 [==============================] - 0s 43ms/step - loss: 0.0602 - accuracy: 0.7841

Epoch 00080: loss did not improve from 0.04527
Epoch 81/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0513 - accuracy: 0.8272

Epoch 00081: loss did not improve from 0.04527
Epoch 82/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0517 - accuracy: 0.8148

Epoch 00082: loss did not improve from 0.04527
Epoch 83/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0604 - accuracy: 0.8148

Epoch 00083: loss

11/11 [==============================] - 0s 41ms/step - loss: 0.0726 - accuracy: 0.7901

Epoch 00106: loss did not improve from 0.04527
Epoch 107/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0771 - accuracy: 0.7654

Epoch 00107: loss did not improve from 0.04527
Epoch 108/200
11/11 [==============================] - 0s 44ms/step - loss: 0.0881 - accuracy: 0.7531

Epoch 00108: loss did not improve from 0.04527
Epoch 109/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0722 - accuracy: 0.7654

Epoch 00109: loss did not improve from 0.04527
Epoch 110/200
11/11 [==============================] - 0s 40ms/step - loss: 0.1075 - accuracy: 0.7037

Epoch 00110: loss did not improve from 0.04527
Epoch 111/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0662 - accuracy: 0.8025

Epoch 00111: loss did not improve from 0.04527
Epoch 112/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0579 - accuracy: 0.8642

Epoch 00112

11/11 [==============================] - 0s 42ms/step - loss: 0.0913 - accuracy: 0.7654

Epoch 00135: loss did not improve from 0.04527
Epoch 136/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0897 - accuracy: 0.7407

Epoch 00136: loss did not improve from 0.04527
Epoch 137/200
11/11 [==============================] - 0s 43ms/step - loss: 0.0731 - accuracy: 0.7531

Epoch 00137: loss did not improve from 0.04527
Epoch 138/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0686 - accuracy: 0.8025

Epoch 00138: loss did not improve from 0.04527
Epoch 139/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0551 - accuracy: 0.8889

Epoch 00139: loss did not improve from 0.04527
Epoch 140/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0637 - accuracy: 0.7531

Epoch 00140: loss did not improve from 0.04527
Epoch 141/200
11/11 [==============================] - 0s 45ms/step - loss: 0.0655 - accuracy: 0.7901

Epoch 00141

Epoch 164/200
11/11 [==============================] - 0s 45ms/step - loss: 0.0743 - accuracy: 0.8025

Epoch 00164: loss did not improve from 0.04334
Epoch 165/200
11/11 [==============================] - 0s 47ms/step - loss: 0.0626 - accuracy: 0.8519

Epoch 00165: loss did not improve from 0.04334
Epoch 166/200
11/11 [==============================] - 0s 46ms/step - loss: 0.0901 - accuracy: 0.7901

Epoch 00166: loss did not improve from 0.04334
Epoch 167/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0580 - accuracy: 0.8148

Epoch 00167: loss did not improve from 0.04334
Epoch 168/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0664 - accuracy: 0.8068

Epoch 00168: loss did not improve from 0.04334
Epoch 169/200
11/11 [==============================] - 0s 45ms/step - loss: 0.0759 - accuracy: 0.7901

Epoch 00169: loss did not improve from 0.04334
Epoch 170/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0873 - accuracy: 0.740

11/11 [==============================] - 0s 43ms/step - loss: 0.0529 - accuracy: 0.8068

Epoch 00193: loss did not improve from 0.04334
Epoch 194/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0558 - accuracy: 0.8272

Epoch 00194: loss did not improve from 0.04334
Epoch 195/200
11/11 [==============================] - 0s 40ms/step - loss: 0.0667 - accuracy: 0.8148

Epoch 00195: loss did not improve from 0.04334
Epoch 196/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0692 - accuracy: 0.7654

Epoch 00196: loss did not improve from 0.04334
Epoch 197/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0631 - accuracy: 0.8148

Epoch 00197: loss did not improve from 0.04334
Epoch 198/200
11/11 [==============================] - 0s 41ms/step - loss: 0.0694 - accuracy: 0.8519

Epoch 00198: loss did not improve from 0.04334
Epoch 199/200
11/11 [==============================] - 0s 42ms/step - loss: 0.0752 - accuracy: 0.7654

Epoch 00199

In [41]:
print("Total Time - ",stop-start)

Total Time -  1021.3174891478848


In [42]:
print((stop-start) / 3600)

0.2836993025410791


In [43]:
len(y)

10

In [44]:
y_org = []
for val in y:
    if val[0]:
        y_org.append(0)
    elif val[1]:
        y_org.append(1)
    elif val[2]:
        y_org.append(2)

In [45]:

len(y_org)

10

In [46]:
count = 0
for i in range(len(y_org)):
     if y_org[i] == y_hat[i]:
            count += 1
acc = int((count / len(y_org) ) * 100)
acc

100